# 09 Fetch ICCWD images to Drive

ICCWD (Kireev, Crețu et al., 2025; labels CC BY 4.0) gives 10,000 web image URLs with a manual child / no-child label. The images are fetched from their original URLs with **the dataset authors' own script** (spring-epfl/iccwd, pinned commit), which checks every file against the authors' perceptual hash (PDQ; mismatches are deleted) and SHA-256 (exact match, reported).

Ethics (CLAUDE.md Decision Log, 19 and 24 Nov 2026): images go to Drive only, never to git; this notebook never displays an image or a caption; only aggregate counts are printed and saved. CPU runtime is enough. Downloads go straight into the Drive folder, so if the runtime disconnects, just run all cells again: files already there are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, hashlib, subprocess, pathlib, collections
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/robust_child_filtering_model')
OUT = DRIVE_ROOT / 'iccwd'; IMG = OUT / 'images'
IMG.mkdir(parents=True, exist_ok=True)
CSV_REV = '9f76b200c40a8a88c19dc17af8b842581f10f5f5'
CSV_SHA = '15f1d0142aac67740d6f41a233fe157d580c256f16705fd5c86cdfa8209c694e'
SCRIPT_COMMIT = 'ee6695fa063655039346a06d127a347321887047'

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
print('images already on Drive:', sum(1 for _ in IMG.iterdir()))

## 1. Pinned label file and the authors' script

In [ ]:
!pip -q install pdqhash
from huggingface_hub import hf_hub_download
csv_path = hf_hub_download('amcretu/iccwd', 'image_caption_children_in_the_wild_dataset.csv',
                           repo_type='dataset', revision=CSV_REV)
assert sha256(csv_path) == CSV_SHA, 'ICCWD label file does not match the pinned version'
os.chdir('/content')
!rm -rf /content/iccwd_code
!git clone -q https://github.com/spring-epfl/iccwd.git /content/iccwd_code
!git -C /content/iccwd_code checkout -q {SCRIPT_COMMIT}
!git -C /content/iccwd_code log --oneline -1

## 2. Download and verify (authors' script, unchanged; can take one to a few hours)

In [ ]:
os.chdir(OUT)   # the script writes errors.txt to the working directory
!python /content/iccwd_code/dataset_download_images_verify.py --folder={IMG} --dataframe={csv_path}

## 3. Aggregate counts only (no images, captions or URLs)

In [ ]:
import pandas as pd
df = pd.read_csv(csv_path)
df['file'] = [IMG / f'{i}.jpg' for i in range(len(df))]
df['present'] = [f.exists() for f in df.file]
df['sha_exact'] = [r.present and r.sha256_hash != '-' and sha256(r.file) == r.sha256_hash for r in df.itertuples()]
summary = {
    'dataset': 'amcretu/iccwd', 'csv_revision': CSV_REV, 'csv_sha256': CSV_SHA,
    'script': 'spring-epfl/iccwd dataset_download_images_verify.py', 'script_commit': SCRIPT_COMMIT,
    'n_listed': int(len(df)), 'n_kept_pdq_verified': int(df.present.sum()), 'n_sha_exact': int(df.sha_exact.sum()),
    'by_label_listed': df.label.value_counts().to_dict(),
    'by_label_kept': df[df.present].label.value_counts().to_dict(),
    'kept_bytes': int(sum(f.stat().st_size for f in df.file[df.present])),
}
print(json.dumps(summary, indent=1))
(DRIVE_ROOT / 'results' / 'iccwd').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'results' / 'iccwd' / 'fetch_summary.json').write_text(json.dumps(summary, indent=1))
print('saved results/iccwd/fetch_summary.json (aggregate counts only)')